In [5]:
!pip install requests beautifulsoup4 ollama

Defaulting to user installation because normal site-packages is not writeable
  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\alice\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import ollama
import re
import os

# ==========================================
# [Configuration] 모델 및 설정
# ==========================================
MODEL_SMART = "qwen2.5:7b"

# ==========================================
# [Part 1] Scraping & Cleaning Helpers
# ==========================================

def extract_date(text):
    """텍스트에서 날짜 추출 (예: December 11, 2025)"""
    months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    date_pattern = rf'({months}\s+\d{{1,2}},\s+\d{{4}})'
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    return None

def extract_relevant_content(text):
    """[핵심] LATEST DEVELOPMENTS 이후 내용에서 광고/불필요 섹션 정교하게 제거"""
    
    article_date = extract_date(text)
    
    start_marker = "LATEST DEVELOPMENTS"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        text = text[start_idx:]
    
    exclude_markers = [
        "PRESENTED BY", "TOGETHER WITH", "AI TRAINING",
        "Trending AI Tools", "Community AI workflows",
        "Highlights: News, Guides & Events", "That's it for today!"
    ]
    
    include_markers = ["Everything else in AI today"]
    
    lines = text.split('\n')
    result_lines = []
    
    if article_date:
        result_lines.append(f"Date: {article_date}")
        result_lines.append("")
    
    skip_section = False
    
    for line in lines:
        stripped_line = line.strip()
        
        should_include = False
        for marker in include_markers:
            if marker in stripped_line:
                should_include = True
                skip_section = False
                break
        
        if should_include:
            result_lines.append(line)
            continue
        
        should_skip = False
        for marker in exclude_markers:
            if stripped_line.startswith(marker) or marker in stripped_line:
                should_skip = True
                skip_section = True
                break
        
        if should_skip:
            continue
            
        if stripped_line.isupper() and len(stripped_line) > 3 and ' ' in stripped_line:
            is_exclude = any(marker in stripped_line for marker in exclude_markers)
            if not is_exclude:
                skip_section = False
        
        if not skip_section:
            result_lines.append(line)
    
    return '\n'.join(result_lines)

def clean_text_structure(text):
    """HTML 잔여물 및 URL 파라미터 청소"""
    text = re.sub(r'(&utm_source=[^)\s]*)', '', text)
    text = re.sub(r'(\?utm_source=[^)\s]*)', '', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    return text.strip()

def get_links_from_archive(page_num=1):
    url = f"https://robotnews.therundown.ai/archive?page={page_num}"
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.content, 'html.parser')
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/p/' in href and 'archive' not in href:
                full_link = href if href.startswith('http') else f"https://robotnews.therundown.ai{href}"
                if full_link not in links: links.append(full_link)
        return links
    except Exception as e:
        print(f"Error fetching archive: {e}")
        return []

def scrape_article(url):
    """URL에서 기사 내용 스크래핑 및 정제"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for script in soup(["script", "style", "nav", "footer"]): 
            script.decompose()
            
        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True): 
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")
                
        raw_text = soup.get_text(separator='\n', strip=True)
        processed_content = extract_relevant_content(raw_text)
        final_text = clean_text_structure(processed_content)
        article_date = extract_date(raw_text) or "Unknown Date"
        
        return {
            "full_text": final_text, 
            "url": url, 
            "date": article_date
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

def clean_json_output(response_text):
    text = response_text.strip()
    if text.startswith('```json'): text = text[7:]
    elif text.startswith('```'): text = text[3:]
    if text.endswith('```'): text = text[:-3]
    return text.strip()

# ==========================================
# [Part 2] Agents (Simplified - No Refiner/Critic)
# ==========================================

def agent_extractor(full_text, date):
    print("  [1] Extraction Agent running...")
    prompt = f"""
    You are an expert AI News Data Extractor. 
    Your goal is to split the provided text into individual news items based on specific structural rules.

    ## Context Info
    Article Date: {date}

    ## Structural Rules (CRITICAL)
    1. **Type 1: Main Articles**
       - Look for section headers (e.g., "NOUS RESEARCH", "AI WEARABLES").
       - The Source URL usually appears in parentheses right after the title.
    
    2. **Type 2: Brief Articles**
       - Look under the section "Everything else in AI today".
       - Format: "Company/Topic [verb] (URL) description".
       - The Source URL appears immediately after the title/verb.
       - Extract EACH brief item as a separate entry.

    ## Extraction Rules
    - **source_name**: Extract the publisher/company name (e.g., "Microsoft", "DeepSeek").
    - **source_url**: Extract the specific link to the article/paper. DO NOT use the newsletter's archive link.

    ## Output Format
    Return a JSON Array ONLY. No markdown.
    [
      {{
        "raw_title": "Original Title",
        "raw_content": "Full content text of this item any excluding url",
        "source_url": "https://extracted-url.com",
        "source_name": "Publisher Name"
      }}
    ]

    ## Text to Process:
    {full_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.1, "num_predict": 4096})
    try: 
        return json.loads(clean_json_output(response['response']))
    except: 
        return []

def agent_summarizer_en(item_text):
    prompt = f"""
    You are an AI News Summarizer.
    
    ## Task
    Summarize the provided news text into English.

    ## Constraints
    1. Length: STRICTLY 2-3 sentences.
    2. Content: Focus on the 'who', 'what', and 'why'.
    3. Style: Professional and objective.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response'].strip()

def agent_summarizer_kr(item_text):
    prompt = f"""
    You are an expert AI Translator and Summarizer.
    
    ## Task
    Translate and summarize the provided news text into Korean.

    ## Constraints
    1. Length: 2-3 sentences.
    2. Style: Professional AI news style (해요체).
    3. Content: Capture the key technical facts accurately.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response'].strip()

def agent_classifier(item_text):
    prompt = f"""
    You are an AI Content Classifier. Classify the text using ONLY the allowed lists below.
    
    ## Allowed Categories (Select all that apply)
    - "Business", "Tech/AI", "Healthcare/Science", "Entertainment/Creative", 
    - "Education", "Society/Policy", "Hardware", "Lifestyle"
    
    ## Allowed ProductServices (Select all that apply)
    - "Text AI", "Image AI", "Video AI", "Voice AI", "Agent AI", 
    - "Automation AI", "Multimodal AI", "Vibe Coding AI", "Robotics"
    
    ## Allowed CoreElements (Select all that apply)
    - "Data", "Algorithm", "Compute", "Safety/Ethics"
    
    ## Task
    1. Analyze the text.
    2. Extract 3-5 lowercase keywords.
    3. Select relevant tags from the lists above.
    
    ## Output JSON Format
    {{
      "categories": ["Category1", ...],
      "productServices": ["Service1", ...],
      "coreElements": ["Element1", ...],
      "keywords": ["tag1", "tag2", "tag3"]
    }}
    
    Text to Classify:
    {item_text}
    """
    
    try:
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  
            options={"temperature": 0.1}
        )
        
        raw_text = response['response']
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except json.JSONDecodeError as e:
        print(f"JSON Parsing Error: {e}")
        return {"categories": [], "productServices": [], "coreElements": [], "keywords": []}
    except Exception as e:
        print(f"General Error: {e}")
        return {"categories": [], "productServices": [], "coreElements": [], "keywords": []}

# ==========================================
# [Part 3] Orchestration Engine (Simplified)
# ==========================================

def process_single_news_item(item, article_date):
    """
    간소화된 버전: Refiner/Critic 없이 직접 결과 생성
    기존 6-8번 LLM 호출 → 3번 호출
    """
    print(f"  Processing item: {item.get('raw_title', 'No Title')[:30]}...")
    
    # 1. 영어 요약
    print("    - Summarizing (EN)...")
    summary_en = agent_summarizer_en(item['raw_content'])

    # 2. 한국어 요약
    print("    - Summarizing (KR)...")
    summary_kr = agent_summarizer_kr(item['raw_content'])

    # 3. 분류
    print("    - Classifying...")
    classification = agent_classifier(item['raw_content'])
    
    # 4. 직접 결과물 구성 (Refiner 없이)
    eng_output = {
        "title": item.get('raw_title', 'No Title'),
        "summary": summary_en,
        "source": item.get('source_name', ''),
        "sourceUrl": item.get('source_url', ''),
        "publishedDate": article_date,
        "likes": 0,
        "shareCount": 0,
        "popularityScore": 0,
        "categories": classification.get('categories', []),
        "productServices": classification.get('productServices', []),
        "coreElements": classification.get('coreElements', []),
        "searchKeywords": classification.get('keywords', [])
    }
    
    # 한국어 키워드 변환 (간단한 매핑)
    kr_keywords = []
    keyword_map = {
        'ai': 'AI', 'llm': 'LLM', 'model': '모델', 'training': '학습',
        'reasoning': '추론', 'math': '수학', 'code': '코드', 'agent': '에이전트',
        'robot': '로봇', 'image': '이미지', 'video': '비디오', 'voice': '음성',
        'data': '데이터', 'algorithm': '알고리즘', 'compute': '컴퓨팅',
        'safety': '안전', 'ethics': '윤리', 'business': '비즈니스',
        'healthcare': '헬스케어', 'education': '교육', 'hardware': '하드웨어'
    }
    
    for kw in classification.get('keywords', []):
        kw_lower = kw.lower()
        if kw_lower in keyword_map:
            kr_keywords.append(keyword_map[kw_lower])
        else:
            kr_keywords.append(kw)  # 매핑 없으면 원본 유지
    
    kor_output = {
        "title": item.get('raw_title', 'No Title'),  # 원본 제목 유지
        "summary": summary_kr,
        "source": item.get('source_name', ''),
        "sourceUrl": item.get('source_url', ''),
        "publishedDate": article_date,
        "likes": 0,
        "shareCount": 0,
        "popularityScore": 0,
        "categories": classification.get('categories', []),
        "productServices": classification.get('productServices', []),
        "coreElements": classification.get('coreElements', []),
        "searchKeywords": kr_keywords
    }
    
    print(f"    ✅ Done")
    return eng_output, kor_output


def main_pipeline(start_page=3, end_page=5):
    """
    간소화된 메인 파이프라인
    """
    print("=== AI News Pipeline Started (Simplified Mode) ===")
    print(f"Model: {MODEL_SMART}")
    print(f"Mode: No Refiner/Critic (3 LLM calls per item)")
    
    # 출력 디렉토리 생성
    os.makedirs("data/data_en", exist_ok=True)
    os.makedirs("data/data_ko", exist_ok=True)
    
    total_items = 0
    start_time = time.time()
    
    for page_num in range(start_page, end_page + 1):
        page_start = time.time()
        print(f"\n{'='*60}")
        print(f"Processing Page {page_num}")
        print('='*60)
        
        # 1. 링크 수집
        links = get_links_from_archive(page_num=page_num)
        
        if not links:
            print(f"No links found on page {page_num}. Skipping.")
            continue

        # 결과를 모을 리스트 (페이지마다 초기화)
        all_english_results = []
        all_korean_results = []

        print(f"Found {len(links)} newsletters on page {page_num}. Starting batch process...")

        for i, target_url in enumerate(links):
            print(f"\n[{i+1}/{len(links)}] Processing Newsletter: {target_url}")
            
            # 2. 스크래핑
            article_data = scrape_article(target_url)
            if not article_data: 
                print("  -> Scraping failed. Skipping.")
                continue

            # 3. 추출 (Extractor)
            raw_items = agent_extractor(article_data['full_text'], article_data['date'])
            print(f"  -> Extracted {len(raw_items)} items.")
            
            # 4. 뉴스 아이템별 처리
            for idx, item in enumerate(raw_items):
                try:
                    eng_result, kor_result = process_single_news_item(item, article_data['date'])
                    if eng_result:
                        all_english_results.append(eng_result)
                    if kor_result:
                        all_korean_results.append(kor_result)
                except Exception as e:
                    print(f"    Error processing item {idx}: {e}")
                    continue

        # 페이지별 파일 저장
        filename_en = f"data/data_en/robot_runtime_news_english_p{page_num}.json"
        filename_kr = f"data/data_ko/robot_runtime_news_korean_p{page_num}.json"
        
        with open(filename_en, 'w', encoding='utf-8') as f:
            json.dump(all_english_results, f, ensure_ascii=False, indent=2)
            
        with open(filename_kr, 'w', encoding='utf-8') as f:
            json.dump(all_korean_results, f, ensure_ascii=False, indent=2)
        
        page_time = time.time() - page_start
        total_items += len(all_english_results)
            
        print(f"\n✅ Page {page_num} Completed: {len(all_english_results)} items in {page_time/60:.1f} min")
        print(f"   Saved to: {filename_en} & {filename_kr}")
    
    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"=== All Pages Processed ===")
    print(f"Total items: {total_items}")
    print(f"Total time: {total_time/3600:.1f} hours")
    if total_items > 0:
        print(f"Average: {total_time/total_items:.1f} sec/item")


if __name__ == "__main__":
    main_pipeline(start_page=3, end_page=5)

=== AI News Pipeline Started (Simplified Mode) ===
Model: qwen2.5:7b
Mode: No Refiner/Critic (3 LLM calls per item)

Processing Page 3
Found 12 newsletters on page 3. Starting batch process...

[1/12] Processing Newsletter: https://robotnews.therundown.ai/p/tesla-offers-musk-1-trillion-pay-raise
  [1] Extraction Agent running...
